# Redox Tower — Interactive

In [87]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')


In [88]:
R    = 8.314e-3   # kJ/(mol K)
F    = 96.485     # kJ/(mol V)
LN10 = np.log(10)

def _rtf(T_K=298.15):
    return R * T_K / F  # V

def E_O2_H2O(pH, **kw):
    return 1000 * (1.229 + _rtf()/4*np.log(0.21) - _rtf()*LN10*pH)

def E_CO2_Glc(pH, log_pCO2, **kw):
    pCO2 = 10**log_pCO2
    return 1000 * (-0.016 + _rtf()/24*np.log(pCO2**6/1e-3) - _rtf()*LN10*pH)

def E_NO3_N2(pH, **kw):
    return 1000 * (1.248 + _rtf()/10*np.log(1e-3**2) - (6/5)*_rtf()*LN10*pH)

def E_NO3_NO2(pH, **kw):
    return 1000 * (0.844 + _rtf()/2*np.log(1.0) - _rtf()*LN10*pH)

def E_NO3_NH4(pH, **kw):
    return 1000 * (0.878 + _rtf()/8*np.log(1e-3/1e-4) - (10/8)*_rtf()*LN10*pH)

def E_Fe(pH, **kw):
    return 1000 * (0.314 + _rtf()*np.log(1e-5) - _rtf()*LN10*pH)

def E_SO4_HS(pH, log_cSO4, **kw):
    cSO4 = 10**log_cSO4
    return 1000 * (0.249 + _rtf()/8*np.log(cSO4/1e-4) - (9/8)*_rtf()*LN10*pH)

def E_CO2_CH4(pH, log_pCO2, log_pCH4, **kw):
    pCO2, pCH4 = 10**log_pCO2, 10**log_pCH4
    return 1000 * (0.169 + _rtf()/8*np.log(pCO2/pCH4) - _rtf()*LN10*pH)

def E_NAD(pH, **kw):
    return 1000 * (-0.113 + _rtf()/2*np.log(1.0) - (1/2)*_rtf()*LN10*pH)

def E_H2(pH, log_pH2, **kw):
    return 1000 * (-_rtf()*LN10*pH - _rtf()/2*np.log(10**log_pH2))

def E_CO2_Ac(pH, log_pCO2, **kw):
    pCO2 = 10**log_pCO2
    return 1000 * (0.072 + _rtf()/8*np.log(pCO2**2/5e-3) - (7/8)*_rtf()*LN10*pH)

def E_CO2_Prop(pH, log_pCO2, **kw):
    pCO2 = 10**log_pCO2
    return 1000 * (0.132 + _rtf()/6*np.log(5e-3*pCO2/1e-3) - _rtf()*LN10*pH)

def E_CO2_But(pH, log_pCO2, **kw):
    pCO2 = 10**log_pCO2
    return 1000 * (0.113 + _rtf()/20*np.log(pCO2**4/1e-3) - (19/20)*_rtf()*LN10*pH)

def E_Crot_But(pH, **kw):
    return 1000 * (0.404 + _rtf()/2*np.log(1.0) - _rtf()*LN10*pH)

def E_Pyr_Lac(pH, **kw):
    return 1000 * (0.229 + _rtf()/2*np.log(1.0) - _rtf()*LN10*pH)

def E_AcAld_EtOH(pH, **kw):
    return 1000 * (0.217 + _rtf()/2*np.log(1e-4/1e-3) - _rtf()*LN10*pH)

HALF_RXNS = [
    dict(id='o2',      label='O2 / H2O',         color='#E65100', E_std=+816, col=0, func=E_O2_H2O),
    dict(id='no3n2',   label='NO3 / N2',          color='#558B2F', E_std=+750, col=0, func=E_NO3_N2),
    dict(id='no3no2',  label='NO3 / NO2',         color='#827717', E_std=+430, col=0, func=E_NO3_NO2),
    dict(id='glc',     label='CO2 / Glc',         color='#1565C0', E_std=-430, col=0, func=E_CO2_Glc),
    dict(id='no3nh4',  label='NO3 / NH4+',        color='#00838F', E_std=+360, col=1, func=E_NO3_NH4),
    dict(id='fe',      label='Fe(III)/Fe(II)',    color='#8D6E63', E_std=-100, col=1, func=E_Fe),
    dict(id='hs',      label='SO4 / HS-',         color='#6A1B9A', E_std=-217, col=1, func=E_SO4_HS),
    dict(id='ch4',     label='CO2 / CH4',         color='#2E7D32', E_std=-244, col=1, func=E_CO2_CH4),
    dict(id='crotbut', label='CrCoA / ButCoA',   color='#FF7043', E_std=-10,  col=2, func=E_Crot_But),
    dict(id='pyrlac',  label='Pyr / Lac',         color='#AD1457', E_std=-185, col=2, func=E_Pyr_Lac),
    dict(id='etoh',    label='AcAld / EtOH',      color='#00695C', E_std=-197, col=2, func=E_AcAld_EtOH),
    dict(id='co2ac',   label='CO2 / Ac-',         color='#B71C1C', E_std=-290, col=2, func=E_CO2_Ac),
    dict(id='co2but',  label='CO2 / But-',        color='#6200EA', E_std=-280, col=2, func=E_CO2_But),
    dict(id='prop',    label='CO2 / Prop-',       color='#F9A825', E_std=-282, col=2, func=E_CO2_Prop),
    dict(id='nad',     label='NAD+ / NADH',       color='#6D4C41', E_std=-320, col=2, func=E_NAD),
    dict(id='h2',      label='H+ / H2',           color='#37474F', E_std=-414, col=2, func=E_H2),
]

# Columns shifted right to leave room for reaction arrows on the left
COL_DEFS = {
    0: dict(x0=0.11, x1=0.24, tx=0.25, hx=0.175, title='Aerobic'),
    1: dict(x0=0.38, x1=0.51, tx=0.52, hx=0.445, title='Anaerobic\nrespiration'),
    2: dict(x0=0.67, x1=0.80, tx=0.81, hx=0.735, title='Fermentation /\nsyntrophic'),
}


In [89]:
# ── Reactions ────────────────────────────────────────────────────────────────
REACTIONS = [
    dict(id='aerobic',  label='Aerobic resp.',          color='#E65100', ne=24,
         donor='glc',   acceptor='o2',
         eq_don='Glucose + 6H2O → 6CO2 + 24H⁺ + 24e⁻',
         eq_acc='6O2 + 24H⁺ + 24e⁻ → 12H2O',
         eq_net='Glucose + 6O2 → 6CO2 + 6H2O'),
    dict(id='denitrif', label='Denitrification',         color='#558B2F', ne=24,
         donor='glc',   acceptor='no3n2',
         eq_don='Glucose + 6H2O → 6CO2 + 24H⁺ + 24e⁻',
         eq_acc='2.4 NO3⁻ + 28.8H⁺ + 24e⁻ → 1.2 N2 + 14.4H2O',
         eq_net='Glucose + 2.4 NO3⁻ → 6CO2 + 1.2 N2 + H2O'),
    dict(id='h2meth',   label='H2 methanogenesis',       color='#2E7D32', ne=8,
         donor='h2',    acceptor='ch4',
         eq_don='4H2 → 8H⁺ + 8e⁻',
         eq_acc='CO2 + 8H⁺ + 8e⁻ → CH4 + 2H2O',
         eq_net='4H2 + CO2 → CH4 + 2H2O'),
    dict(id='h2sr',     label='H2 sulfate reduction',    color='#6A1B9A', ne=8,
         donor='h2',    acceptor='hs',
         eq_don='4H2 → 8H⁺ + 8e⁻',
         eq_acc='SO4²⁻ + 9H⁺ + 8e⁻ → HS⁻ + 4H2O',
         eq_net='4H2 + SO4²⁻ → HS⁻ + H⁺ + 4H2O'),
    dict(id='saob',     label='Syntrophic acetate ox.',  color='#B71C1C', ne=8,
         donor='co2ac', acceptor='h2',
         eq_don='CH3COO⁻ + 2H2O → 2CO2 + 7H⁺ + 8e⁻',
         eq_acc='8H⁺ + 8e⁻ → 4H2',
         eq_net='CH3COO⁻ + 3H2O → 4H2 + 2HCO3⁻'),
    dict(id='chain',    label='Chain elongation',        color='#FF7043', ne=2,
         donor='pyrlac', acceptor='crotbut',
         eq_don='Lactate → Pyruvate + 2H⁺ + 2e⁻',
         eq_acc='Crotonyl-CoA + 2H⁺ + 2e⁻ → Butyryl-CoA',
         eq_net='Lactate + Crotonyl-CoA → Pyruvate + Butyryl-CoA'),
]

HR_COL        = {hr['id']: hr['col'] for hr in HALF_RXNS}
arrow_artists = {}
selected_rxn  = [None]
popup_ann     = [None]

plt.ioff()
fig, ax = plt.subplots(figsize=(4.5, 4.5))
plt.subplots_adjust(left=0.14, right=0.99, top=0.91, bottom=0.05)
Y_MIN, Y_MAX = -800, 950

def remove_popup():
    if popup_ann[0] is not None:
        try: popup_ann[0].remove()
        except Exception: pass
        popup_ann[0] = None

def show_popup(rxn_id, E_don, E_acc, x_click, y_click):
    remove_popup()
    rxn   = next(r for r in REACTIONS if r['id'] == rxn_id)
    dE    = E_acc - E_don
    dG    = -rxn['ne'] * F * dE / 1000
    sign  = '▼ exergonic' if dG < 0 else '▲ endergonic'
    txt   = (
        f"{rxn['label']}\n"
        f"{'─'*30}\n"
        f"ox:  {rxn['eq_don']}\n"
        f"red: {rxn['eq_acc']}\n"
        f"{'─'*30}\n"
        f"E_donor    = {E_don:+.0f} mV\n"
        f"E_acceptor = {E_acc:+.0f} mV\n"
        f"ΔE         = {dE:+.0f} mV\n"
        f"ΔG (n={rxn['ne']:2d})  = {dG:+.1f} kJ/mol  {sign}"
    )
    popup_ann[0] = ax.annotate(
        txt, xy=(x_click, y_click), xytext=(x_click + 0.03, y_click),
        fontsize=6.5, fontfamily='monospace', va='center', ha='left',
        bbox=dict(boxstyle='round,pad=0.4', fc='#FFFDE7', ec=rxn['color'], lw=1.5, alpha=0.97),
        arrowprops=dict(arrowstyle='->', color=rxn['color'], lw=1.0),
        zorder=20
    )

def on_pick(event):
    for rxn_id, (line, E_don, E_acc, xb, ymid) in arrow_artists.items():
        if event.artist is line:
            if selected_rxn[0] == rxn_id:
                selected_rxn[0] = None;  remove_popup()
            else:
                selected_rxn[0] = rxn_id
                show_popup(rxn_id, E_don, E_acc, xb, ymid)
            fig.canvas.draw_idle()
            return

fig.canvas.mpl_connect('pick_event', on_pick)

def redraw(pH, log_pH2, log_pCO2, log_pCH4, log_cSO4):
    remove_popup()
    ax.cla()
    arrow_artists.clear()
    kw    = dict(pH=pH, log_pH2=log_pH2, log_pCO2=log_pCO2,
                 log_pCH4=log_pCH4, log_cSO4=log_cSO4)
    E_now = {hr['id']: hr['func'](**kw) for hr in HALF_RXNS}

    for cd in COL_DEFS.values():
        ax.text(cd['hx'], Y_MAX - 10, cd['title'],
                ha='center', va='top', fontsize=6.5, fontweight='bold', color='#444',
                bbox=dict(boxstyle='round,pad=0.2', fc='#f0f0f0', ec='#bbb', lw=0.8))

    for hr in HALF_RXNS:
        E  = E_now[hr['id']]
        c  = hr['color']
        cd = COL_DEFS[hr['col']]
        ax.plot([cd['x0'], cd['x1']], [hr['E_std'], hr['E_std']], color=c, lw=1.5,
                linestyle='--', alpha=0.35, zorder=2)
        ax.plot([cd['x0'], cd['x1']], [E, E], color=c, lw=1.5,
                solid_capstyle='butt', alpha=0.90, zorder=3)
        ax.text(cd['tx'], E, hr['label'],
                fontsize=6.5, color=c, fontweight='bold', va='center', ha='left')

    active = [r for r in REACTIONS if rxn_checks[r['id']].value]
    same_col_n = {}
    for rxn in active:
        E_don  = E_now[rxn['donor']];  E_acc = E_now[rxn['acceptor']]
        col_d  = HR_COL[rxn['donor']]; col_a = HR_COL[rxn['acceptor']]
        cd_d   = COL_DEFS[col_d];      cd_a  = COL_DEFS[col_a]
        is_sel = rxn['id'] == selected_rxn[0]
        lw     = 2.0 if is_sel else 1.2

        if col_d == col_a:
            n  = same_col_n.get(col_d, 0);  same_col_n[col_d] = n + 1
            xb = cd_d['x1'] + 0.008 + n * 0.013
            ymid = (E_don + E_acc) / 2
            line, = ax.plot([xb, xb], [E_don, E_acc],
                            color=rxn['color'], lw=lw, alpha=0.85, picker=6, zorder=10)
            ax.annotate('', xy=(xb, E_acc), xytext=(xb, E_don),
                        arrowprops=dict(arrowstyle='->', color=rxn['color'], lw=lw, mutation_scale=8))
        else:
            x_don = (cd_d['x0'] + cd_d['x1']) / 2
            x_acc = (cd_a['x0'] + cd_a['x1']) / 2
            xb    = (x_don + x_acc) / 2;  ymid = (E_don + E_acc) / 2
            line, = ax.plot([x_don, x_acc], [E_don, E_acc],
                            color=rxn['color'], lw=lw, alpha=0.85, picker=6, zorder=10)
            ax.annotate('', xy=(x_acc, E_acc), xytext=(x_don, E_don),
                        arrowprops=dict(arrowstyle='->', color=rxn['color'], lw=lw, mutation_scale=8))

        ax.text(xb, ymid, 'ⓘ', fontsize=6, color=rxn['color'],
                ha='center', va='center', zorder=11,
                bbox=dict(fc='white', ec='none', pad=0.1))
        arrow_artists[rxn['id']] = (line, E_don, E_acc, xb, ymid)
        if is_sel:
            show_popup(rxn['id'], E_don, E_acc, xb, ymid)

    ax.set_xlim(0, 1);  ax.set_ylim(Y_MIN, Y_MAX);  ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_ylabel('Reduction Potential (mV vs SHE)', fontsize=8)
    ax.set_title(
        f"pH={pH:.1f}  H2={10**log_pH2*1e6:.1g} ppm  "
        f"CO2={10**log_pCO2*100:.0f}%  CH4={10**log_pCH4*100:.0f}%  "
        f"SO4={10**log_cSO4*1000:.1f} mM",
        fontsize=7.5, pad=6
    )
    ax.tick_params(labelsize=7)
    ax.grid(axis='y', alpha=0.2, linestyle=':')
    ax.spines[['top', 'right', 'bottom']].set_visible(False)
    fig.canvas.draw_idle()

s = {'description_width': '95px'}
l = widgets.Layout(width='260px')
w_pH   = widgets.FloatSlider(value=7.0,   min=5.5,  max=8.5,  step=0.1,  description='pH',         style=s, layout=l, readout_format='.1f')
w_lh2  = widgets.FloatSlider(value=-3.5,  min=-7.0, max=-1.0, step=0.25, description='log(p_H2)',  style=s, layout=l, readout_format='.2f')
w_lco2 = widgets.FloatSlider(value=-0.7,  min=-3.0, max=0.0,  step=0.25, description='log(p_CO2)',style=s, layout=l, readout_format='.2f')
w_lch4 = widgets.FloatSlider(value=-1.0,  min=-4.0, max=0.0,  step=0.25, description='log(p_CH4)',style=s, layout=l, readout_format='.2f')
w_lso4 = widgets.FloatSlider(value=-1.55, min=-4.0, max=0.0,  step=0.25, description='log([SO4])',style=s, layout=l, readout_format='.2f')

def on_change(change):
    redraw(w_pH.value, w_lh2.value, w_lco2.value, w_lch4.value, w_lso4.value)

for w in [w_pH, w_lh2, w_lco2, w_lch4, w_lso4]:
    w.observe(on_change, names='value')

rxn_checks = {}
for r in REACTIONS:
    cb = widgets.Checkbox(value=False, description=r['label'], indent=False,
                          layout=widgets.Layout(width='190px', margin='1px 0'))
    cb.observe(on_change, names='value')
    rxn_checks[r['id']] = cb

rxn_box = widgets.VBox(
    [widgets.HTML('<b style="font-size:9px">Reaction arrows (click ⓘ for details):</b>')]
    + list(rxn_checks.values()),
    layout=widgets.Layout(padding='4px 6px', border='1px solid #ddd', width='200px')
)

top = widgets.HBox(
    [widgets.VBox([w_pH, w_lh2, w_lco2, w_lch4, w_lso4]), rxn_box],
    layout=widgets.Layout(gap='10px', align_items='flex-start')
)

redraw(w_pH.value, w_lh2.value, w_lco2.value, w_lch4.value, w_lso4.value)
display(widgets.VBox([top, fig.canvas], layout=widgets.Layout(gap='4px')))
